In [1]:
# ==============================
# FAKE NEWS DETECTION - FINAL
# LIGHTWEIGHT PIPELINE
# ==============================

import pandas as pd
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


# 1. Load Dataset
data_fake = pd.read_csv("data/Fake.csv")
data_true = pd.read_csv("data/True.csv")


# 2. Add Labels
data_fake["class"] = 0
data_true["class"] = 1


# 3. Remove 10 records kept for manual testing
data_fake = data_fake.iloc[:-10]
data_true = data_true.iloc[:-10]


# 4. Combine Dataset
data_merge = pd.concat([data_fake, data_true], axis=0)

data = data_merge.drop(
    ["title", "subject", "date"],
    axis=1
)

data = data.sample(frac=1, random_state=42).reset_index(drop=True)


# 5. Text Preprocessing
def wordopt(text):
    text = text.lower()
    text = re.sub(r"\[.*?\]", "", text)
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"<.*?>+", "", text)
    text = re.sub(r"[%s]" % re.escape(string.punctuation), "", text)
    text = re.sub(r"\n", "", text)
    text = re.sub(r"\w*\d\w*", "", text)
    return text


data["text"] = data["text"].apply(wordopt)


# 6. Input and Target
x = data["text"]
y = data["class"]


# 7. Train-Test Split
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)


# 8. TF-IDF
# Limit features to keep the model fast
vectorization = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2)
)

xv_train = vectorization.fit_transform(x_train)
xv_test = vectorization.transform(x_test)


print("x_train:", x_train.shape)
print("x_test:", x_test.shape)
print("xv_train:", xv_train.shape)
print("xv_test:", xv_test.shape)


# 9. Logistic Regression
LR = LogisticRegression(
    max_iter=1000
)

LR.fit(xv_train, y_train)


# 10. Evaluation
pred_lr = LR.predict(xv_test)

accuracy = accuracy_score(y_test, pred_lr)

print("\nLogistic Regression Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, pred_lr))


# 11. Manual Testing
def output_lable(n):
    if n == 0:
        return "Fake News"
    elif n == 1:
        return "Not A Fake News"


def manual_testing(news):
    cleaned_news = wordopt(news)

    news_vector = vectorization.transform([cleaned_news])

    prediction = LR.predict(news_vector)

    print("\nPrediction:", output_lable(prediction[0]))


# 12. Manual Test
news = str(input("\nEnter news: "))

manual_testing(news)

x_train: (33658,)
x_test: (11220,)
xv_train: (33658, 30000)
xv_test: (11220, 30000)

Logistic Regression Accuracy: 0.9894830659536542

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5868
           1       0.99      0.99      0.99      5352

    accuracy                           0.99     11220
   macro avg       0.99      0.99      0.99     11220
weighted avg       0.99      0.99      0.99     11220


Prediction: Fake News
